# Figure 3 Dashboard — Stomatal Oscillations Parameter Explorer

Explore how different parameter groups affect stomatal oscillations driven by a 3-step humidity protocol (RH: 80% → 55% → 30%).

**Dashboards:**
1. **Hydraulics** — tissue resistances, elastic moduli, conductance gain
2. **Vstress → ΔPg curve** — stress-sensing sigmoid
3. **ABA Biosynthesis** — synthesis/catabolism kinetics
4. **ABA Signaling** — receptor-kinase cascade
5. **Electrophysiology** — ion channels, pumps

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import io, contextlib, warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import pandas as pd

import panel as pn
pn.extension("bokeh")

print('Imports loaded (matplotlib + Panel).')

## 2. Model Parameters

In [2]:
def read_parameters_oscillations():
    """
    Returns a dictionary of model parameters for the coupled hydropassive-
    hydroactive stomatal conductance model — oscillation scenario.
    
    Key difference from WT transients: φ₃ = 0.05 (vs 20) inverts the V₋
    formula: V₋ = K_d · γ_H² / (γ_A · φ₃), creating a regime where the
    ABA autoregulatory motif exhibits limit-cycle oscillations.
    
    Uses a 3-step RH protocol (80% → 55% → 30%) with shorter simulation
    windows (3 × 100-min periods).
    """
    hyd = {}

    # =========================================================================
    # 1. TISSUE VOLUMES & TURGOR PRESSURES
    # =========================================================================
    # Symplastic water volumes (mmol H₂O / m² leaf area)
    hyd["Vg0"] = 100.0                     # guard cell
    hyd["Vs0"] = 100.0                     # subsidiary cell
    hyd["Vm0"] = 1978.0                    # mesophyll

    hyd["stomatal_density"] = 150e6        # stomata / m² leaf

    # Bulk elastic moduli (MPa) — tissue stiffness
    hyd["epsilong"] = 3.6                  # guard cell
    hyd["epsilons"] = 0.5                  # subsidiary cell
    hyd["epsilonm"] = 4.0                  # mesophyll

    # Initial turgor pressures (MPa)
    hyd["Pg0"] = 2.5                       # guard cell
    hyd["Ps0"] = 1.0                       # subsidiary cell
    hyd["Pm0"] = 2.2                       # mesophyll

    # Initial osmotic pressures (MPa); offset from turgor by 0.1
    hyd["pig0"] = hyd["Pg0"] + 0.1
    hyd["pis0"] = hyd["Ps0"] + 0.1
    hyd["pim0"] = hyd["Pm0"] + 0.1

    # Initial turgor perturbations (MPa)
    hyd["DeltaPim"] = 0.0
    hyd["DeltaPis"] = 0.0
    hyd["DeltaPig"] = 0.0

    # =========================================================================
    # 2. GAS CONSTANTS & HYDRAULIC TRANSPORT
    # =========================================================================
    hyd["R"]    = 8.314                    # universal gas constant (J/mol/K)
    hyd["T"]    = 308.0                    # temperature (K) ≈ 35°C
    hyd["nu_w"] = 1.8e-5                   # molar volume of water (m³/mol)
    hyd["RT_by_nu_by1e6"] = (hyd["R"] * hyd["T"] / hyd["nu_w"]) / 1e6

    # Stomatal conductance equation:  gs = Xi * max(Pg - m*Ps, 0)
    hyd["Xi"] = 295.0                      # conductance gain (mmol/m²/s/MPa)
    hyd["m"]  = 1.71                       # mechanical advantage of subsidiary cells

    # Hydraulic resistances (MPa·s·m²/mmol) — flow = ΔΨ / R
    hyd["Rxm"] = 0.018116                  # xylem → mesophyll
    hyd["Rms"] = 5.0                       # mesophyll → subsidiary cell
    hyd["Rsg"] = 10.0                      # subsidiary cell → guard cell

    # =========================================================================
    # 3. CONDUCTANCE BASELINES & ATMOSPHERIC CONDITIONS
    # =========================================================================
    # Stomatal and cuticular conductance coefficients (mmol/m²/s)
    hyd["gg1"] = 2.9                       # g_g¹ — guard cell (stomatal) pathway
    hyd["ge1"] = 2.9                       # g_e¹ — epidermal (cuticular) pathway
    hyd["gg2"] = 1.0                       # g_g² — guard cell VPD sensitivity
    hyd["ge2"] = 10.0                      # g_e² — epidermal VPD sensitivity

    # Saturation vapor pressure and atmospheric pressure
    hyd["psat"]      = 0.00563             # saturation vapor pressure (MPa) at T
    hyd["patm"]      = 0.101               # atmospheric pressure (MPa)
    hyd["psat_patm"] = hyd["psat"] / hyd["patm"]

    # =========================================================================
    # 4. HUMIDITY PROTOCOL (3-step RH: 80% → 55% → 30%)
    # =========================================================================
    hyd["RH1"] = 80.0                      # pre-step RH (%)
    hyd["RH2"] = 55.0                      # first step RH (%)
    hyd["RH3"] = 30.0                      # second step RH (%)
    hyd["VPD1"] = hyd["psat"] * (1 - hyd["RH1"] / 100)
    hyd["VPD2"] = hyd["psat"] * (1 - hyd["RH2"] / 100)
    hyd["VPD3"] = hyd["psat"] * (1 - hyd["RH3"] / 100)
    hyd["psix_at_RH_step"] = -0.1          # xylem water potential at step (MPa)

    # =========================================================================
    # 5. ABA BIOSYNTHESIS & CATABOLISM
    #    Autoregulatory motif: NCED (positive fb) + CYP707A/A8H (negative fb)
    #    OSCILLATION REGIME: φ₃ = 0.05 → V₋ = K_d·γ_H²/(γ_A·φ₃)
    # =========================================================================
    # Genetic pre-factors (1.0 = wild-type; set to 0 for knockouts)
    hyd["thetaABA"] = 1.0                  # θ_NCED — synthesis (NCED3/5)
    hyd["thetaA8H"] = 1.0                  # θ_A8H  — catabolism (CYP707A)

    hyd["n"]    = 3                        # Hill coefficient (stress cooperativity)
    hyd["kcat"] = 0.25                     # γ_A: A8H catalytic rate (1/s)
    hyd["K2"]   = 2000.0                   # K_d: A8H Michaelis constant (nM)
    hyd["gammac"] = 32 * 9.6e-5            # γ_H: A8H protein decay rate (1/s) ≈ 3.072e-3

    # Half-saturation ratios: K₊ = λ₁·K_d,  K₋ = λ₃·K_d
    hyd["lambda1"] = 0.78
    hyd["lambda3"] = 0.78
    hyd["K1"] = hyd["lambda1"] * hyd["K2"]  # K₊ (nM) — NCED half-saturation
    hyd["K3"] = hyd["lambda3"] * hyd["K2"]  # K₋ (nM) — CYP707A half-saturation

    # Dimensionless rate ratios (φ) — oscillatory regime
    hyd["phi1"] = 0.072                    # stress synthesis scaling
    hyd["phi2"] = 0.55                     # positive feedback strength
    hyd["phi3"] = 0.05                     # catabolism scaling (oscillatory regime)

    # Derived maximal rates (nM/s)
    # NOTE: V₋ formula DIFFERS from WT — divided by φ₃ (not multiplied)
    hyd["Vminus"]  = (hyd["K2"] * hyd["gammac"]**2) / (hyd["kcat"] * hyd["phi3"])
    hyd["Vplus"]   = hyd["phi2"] * hyd["kcat"] * hyd["Vminus"] / hyd["gammac"]
    hyd["Vstress"] = hyd["phi1"] * hyd["kcat"] * hyd["Vminus"] / hyd["gammac"]

    # =========================================================================
    # 6. ABA SIGNALING CASCADE
    #    Bicyclic futile cycle: MAP3K → OST1 → SLAC1
    #    with PP2C dephosphorylation, inhibited by ABA:PYR complex
    # =========================================================================
    # Total protein concentrations (nM)
    hyd["OST1T"]  = 1e3
    hyd["SLAC1T"] = 1e3
    hyd["MAP3KT"] = 1e3
    hyd["PP2CT"]  = 1e3
    hyd["PYRT"]   = 1e3

    # ABA-receptor binding
    hyd["KD"] = 1000.0                     # ABA–PYR dissociation constant (nM)
    hyd["KI"] = 30.0                       # PYR–PP2C inhibition constant (nM)

    # Phosphorylation / dephosphorylation rates (1/s)
    hyd["k1"] = 0.03                       # MAP3K → OST1 (phosphorylation)
    hyd["k2"] = 0.032772                   # PP2C ⊣ OST1 (dephosphorylation)
    hyd["k3"] = 3.75                       # OST1 → SLAC1 (phosphorylation)
    hyd["k4"] = 0.1386                     # PP2C ⊣ SLAC1 (dephosphorylation)

    # Michaelis constants for the four enzymatic steps (nM)
    hyd["Km1"] = 2.5e3
    hyd["Km2"] = 0.0897e3
    hyd["Km3"] = 18.93e3
    hyd["Km4"] = 0.597e3

    # =========================================================================
    # 7. GUARD CELL ELECTROPHYSIOLOGY
    #    GHK ion fluxes through SLAC1 (Cl⁻), GORK (K⁺ out), KAT1 (K⁺ in),
    #    and H⁺-ATPase proton pump
    # =========================================================================
    # Membrane properties
    hyd["Cm"]   = 1e-2                     # membrane capacitance (F/m²)
    hyd["area"] = 20e-10                   # guard cell membrane area (m²)
    hyd["vol_in"]  = hyd["Vg0"] * hyd["nu_w"] * 1e-3 / hyd["stomatal_density"]  # intracellular volume (m³)
    hyd["vol_out"] = 65e-15                # apoplastic volume (m³)
    hyd["F"] = 96485.3329                  # Faraday constant (C/mol)

    # GORK voltage gating
    hyd["delta_GORK"]         = 2.0        # voltage sensitivity parameter
    hyd["V1by2_Kconc_GORK"]   = 150.0      # half-activation K⁺ concentration (mM)

    # KAT1 voltage gating
    hyd["delta_KAT1"]  = 1.6              # voltage sensitivity parameter
    hyd["V1by2_KAT1"]  = -170e-3          # half-activation voltage (V)

    hyd["otherIons"] = 0.0                 # background ion contribution

    # Initial ion concentrations (mM)
    hyd["Clin0"]  = 260.0                  # Cl⁻ intracellular
    hyd["Clout0"] = 22.0                   # Cl⁻ apoplastic
    hyd["Kin0"]   = 210.0                  # K⁺  intracellular
    hyd["Kout0"]  = 20.0                   # K⁺  apoplastic

    # Ion valences
    hyd["zCl"] = -1
    hyd["zK"]  = 1

    # Membrane permeabilities (m/s)
    hyd["PK"]  = 5e-6                      # K⁺ permeability
    hyd["PCl"] = 1e-7                      # Cl⁻ permeability

    # Channel number scaling (relative to baseline)
    hyd["NGORK"]  = 1.0
    hyd["NKAT1"]  = 1.0
    hyd["NSLAC1"] = 1.0

    # Proton concentrations (derived from pH)
    hyd["pHin0"]  = 7.2
    hyd["pHout0"] = 6.5
    hyd["Hin0"]   = 10**(-hyd["pHin0"])  * 1e3   # intracellular H⁺ (mM)
    hyd["Hout0"]  = 10**(-hyd["pHout0"]) * 1e3   # apoplastic H⁺ (mM)
    hyd["zH"] = 1

    # H⁺-ATPase proton pump
    hyd["DeltaGATP"]      = -26000.0       # free energy of ATP hydrolysis (J/mol)
    hyd["E0pump"]         = 1.1033e6       # pump rate constant
    hyd["k1pump"]         = 0.55           # pump kinetic parameter 1
    hyd["k2pump"]         = 2.58e-4        # pump kinetic parameter 2
    hyd["V0Cl"]           = 9.25e4         # SLAC1 Cl⁻ flux scaling
    hyd["ABA_Hpump_Khalf"] = 50.0          # ABA half-inhibition of H⁺-pump (nM)
    hyd["ABA_Hpump_n"]    = 1.0            # Hill coefficient for ABA pump inhibition

    # =========================================================================
    # 8. SIMULATION TIMING (3 × 100-min periods)
    # =========================================================================
    hyd["tstart"] = 0.0                    # simulation start (s)
    hyd["tstep1"] = 6000.0                 # first RH step at 100 min
    hyd["tstep2"] = 12000.0                # second RH step at 200 min
    hyd["tend"]   = 18000.0                # simulation end at 300 min

    return hyd

print("Parameter function defined.")

Parameter function defined.


## 3. Helper Functions

In [3]:
# =============================================================================
# HELPER FUNCTIONS
# Organized by module: Environmental → Hydraulics → ABA → Signaling →
#                      Ion Channels → Membrane Transport
# =============================================================================

# ---- Environmental forcing (3-step RH protocol for oscillations) ------------

def f_RH_3step(t, hyd):
    """3-step relative humidity protocol: RH1 → RH2 → RH3 at tstep1, tstep2."""
    t = np.asarray(t, dtype=float)
    return np.where(t <= hyd["tstep1"], hyd["RH1"],
           np.where(t <= hyd["tstep2"], hyd["RH2"], hyd["RH3"]))

def f_VPD_3step(t, hyd):
    """3-step vapor pressure deficit corresponding to the RH protocol."""
    t = np.asarray(t, dtype=float)
    return np.where(t <= hyd["tstep1"], hyd["VPD1"],
           np.where(t <= hyd["tstep2"], hyd["VPD2"], hyd["VPD3"]))

# ---- Hydraulic transport ----------------------------------------------------

def f_Rox(psix):
    """Overall xylem resistance as a function of xylem water potential.
    
    Empirical sigmoidal fit capturing vulnerability-curve–driven loss of
    xylem hydraulic conductance under drought.
    """
    a, b, c, k = 0.3664, 15.61, -0.7684, 2.958
    return 1.0 / (a + (k - a) / (1.0 + np.exp(-b * (psix - c))))

def f_P_from_psi(psi, epsilon, pi0, P0, DeltaPi_MPa):
    """Convert water potential perturbation (ψ) to turgor pressure (P).
    
    Uses the linearized relation:  P = P0 + ψ / (1 + π0/ε)
    Turgor is clamped to ≥ 0 (no negative turgor).
    """
    return np.maximum(P0 + (psi - DeltaPi_MPa) / (1.0 + pi0/epsilon), 0.0)

def f_SSC_apoplast_potential(psim, psig, psis, gs, gmH2O, pa, hyd, geH2O, ggH2O):
    """Steady-state substomatal cavity (SSC) apoplast water potential.
    
    Balances water inflow from mesophyll, subsidiary, and guard cells against
    evaporative loss through stomata.
    """
    return (psim*gmH2O + psis*geH2O + psig*ggH2O
            - gs*hyd["RT_by_nu_by1e6"]*(hyd["psat"]-pa)/hyd["psat"]) \
            / (gmH2O + geH2O + ggH2O + gs)

# ---- ABA biosynthesis & catabolism ------------------------------------------

def f_Vstress(DeltaPg, hyd):
    """Stress-responsive ABA synthesis rate as a Hill function of turgor loss.
    
    V_stress(ΔPg) = V_stress · |ΔPg|^n / (K_half + |ΔPg|^n)
    
    Uses custom vstress_exponent and vstress_khalf if provided via
    dashboard overrides, otherwise defaults to n=3.25 and K_half=1.4.
    """
    abs_dpg = np.abs(DeltaPg)
    n_hill = hyd.get("vstress_exponent", 3.25)
    k_half = hyd.get("vstress_khalf", 1.4)
    return np.abs(hyd["Vstress"] * abs_dpg**n_hill / (k_half + abs_dpg**n_hill))

# ---- ABA signaling cascade -------------------------------------------------

def f_solve_ABA_PYR_quadratic(ABA, hyd):
    """Solve for free PYR receptor concentration given [ABA].
    
    ABA binds PYR receptors; the ABA:PYR complex inhibits PP2C. Solves
    the quadratic mass-balance for [PYR_free].
    """
    theta_PYR = 1.0 + hyd["KD"] / ABA
    b = hyd["KI"] + (hyd["PP2CT"] - hyd["PYRT"]) / theta_PYR
    c = -hyd["KI"] * hyd["PYRT"] / theta_PYR
    return (-b + np.sqrt(b**2 - 4.0*c)) / 2.0

def f_SLAC1_conc_non_dimensional(Oo, So, hyd):
    """Unphosphorylated SLAC1 fraction from conservation law."""
    K4p   = hyd["K4_nd"] * (1.0 + Oo/hyd["K2_nd"])
    term3 = (hyd["PP2CT"]/hyd["SLAC1T"]) * So / (K4p + So)
    return (1.0 - So - term3) / (1.0 + (hyd["OST1T"]/hyd["SLAC1T"]) * (Oo/hyd["K3_nd"]))

def f_OST1_roots_from_quadratic_non_dimensional(Oo, So, hyd):
    """Unphosphorylated OST1 fraction from conservation law.
    
    Solves the quadratic from OST1 total conservation. Returns positive root.
    """
    S     = f_SLAC1_conc_non_dimensional(Oo, So, hyd)
    theta = 1.0 - Oo * (1.0 + S/hyd["K3_nd"])
    b     = hyd["epsilon1"] + hyd["K1_nd"] - theta
    c     = -theta * hyd["K1_nd"]
    return max((-b + np.sqrt(b**2 - 4.0*c)) / 2.0, 0.0)

# ---- Ion channel gating ----------------------------------------------------

def f_p_open_KAT1_channel(Vmemb, V1by2_KAT1, delta_KAT1, hyd):
    """KAT1 inward-rectifying K⁺ channel open probability (Boltzmann gating)."""
    return 1.0 / (1.0 + np.exp((Vmemb - V1by2_KAT1)*(delta_KAT1*hyd["F"]/(hyd["R"]*hyd["T"]))))

def f_V1by2_GORK(Kout, hyd):
    """GORK half-activation voltage (depends on external K⁺ via Nernst shift)."""
    return 1e-3 * (hyd["F"]/(hyd["R"]*hyd["T"])) * np.log(Kout/hyd["V1by2_Kconc_GORK"])

def f_p_open_GORK_channel(Vmemb, V1by2_GORK, delta_GORK, hyd):
    """GORK outward-rectifying K⁺ channel open probability (Boltzmann gating)."""
    return 1.0 / (1.0 + np.exp(-(Vmemb - V1by2_GORK)*(delta_GORK*hyd["F"]/(hyd["R"]*hyd["T"]))))

# ---- Membrane transport (GHK fluxes, mass balance, H⁺-pump) ----------------

def f_GHK_current_flux(Pion, zion, Vmemb, ionConcIn, ionConcOut, hyd):
    """Goldman-Hodgkin-Katz current density for a single ion species."""
    zeta = Vmemb * hyd["F"] / (hyd["R"] * hyd["T"])
    return (Pion * zion**2 * hyd["F"] * zeta
            * (ionConcIn - ionConcOut*np.exp(-zion*zeta))
            / (1.0 - np.exp(-zion*zeta)))

def f_mass_balance(ionIn, ionIn0, ionOut0, hyd):
    """Apoplastic ion concentration from total ion conservation."""
    Q = ionOut0*hyd["vol_out"] + ionIn0*hyd["vol_in"]
    return (Q - ionIn*hyd["vol_in"]) / hyd["vol_out"]

def f_p_on_Hpump(ABA, hyd):
    """Fraction of H⁺-ATPase pumps active (ABA inhibits the pump).
    
    Hill inhibition: p_on = 1 / (1 + ([ABA]/K_half)^n)
    """
    return 1.0 / (1.0 + (ABA/hyd["ABA_Hpump_Khalf"])**hyd["ABA_Hpump_n"])

def f_HPump_current(p_on_Hpump, Hin, Hout, Vmemb, hyd):
    """H⁺-ATPase electrogenic pump current (two-state kinetic model)."""
    u       = Vmemb * hyd["F"] / (hyd["R"] * hyd["T"])
    kplus1  = hyd["k1pump"] * Hin
    kplus2  = hyd["k2pump"] * u / (1.0 - np.exp(-u))
    kminus1 = hyd["k1pump"] * np.exp(hyd["DeltaGATP"] / (hyd["R"]*hyd["T"]))
    kminus2 = hyd["k2pump"] * Hout * u * np.exp(-u) / (1.0 - np.exp(-u))
    cyc = (kplus1*kplus2 - kminus1*kminus2) / (kplus1 + kplus2 + kminus1 + kminus2)
    return p_on_Hpump * hyd["E0pump"] * cyc

def f_2HClSymport_current(p_on_Hpump, Hin, Hout, Clin, Clout, Vmemb, hyd):
    """2H⁺/Cl⁻ symporter current (secondary active Cl⁻ uptake)."""
    u = Vmemb * hyd["F"] / (hyd["R"] * hyd["T"])
    return p_on_Hpump * hyd["V0Cl"] * (
        Clin*Hin**2*u - Clout*Hout**2*u*np.exp(-u)) / (1.0 - np.exp(-u))

print("Helper functions defined.")


Helper functions defined.


## 4. ODE System

In [4]:
def fullModel_DeltaPg_ode_osc(t, y, hyd):
    """
    Full 10-variable ODE system for the coupled HP-HA stomatal model.
    Oscillation variant using 3-step RH protocol.
    
    State vector y = [ψ_m, ψ_s, ψ_g, ABA, A8H, Oo, So, V_memb, Cl_in, K_in]
    
    Modules:
      1. Hydraulics      — tissue water potentials → turgor pressures → gs
      2. ABA biosynthesis — stress-sensing + autoregulatory motif → [ABA], [A8H]
      3. ABA signaling    — receptor–kinase–phosphatase cascade → OST1*, SLAC1*
      4. Electrophysiology — ion channels + pump → membrane potential + ion fluxes
    """
    # Unpack state variables
    psim, psis, psig, ABA, A8H, Oo, So, Vmemb, Clin, Kin = y

    # ---- Environmental forcing (3-step RH protocol) ----
    RH   = float(f_RH_3step(t, hyd))
    psix = hyd["psix_at_RH_step"]
    pa   = hyd["psat"] * RH / 100.0            # ambient vapor pressure (MPa)

    # =========================================================================
    # MODULE 1: HYDRAULICS
    # =========================================================================
    pm = hyd["psat"] * (1.0 + psim/hyd["RT_by_nu_by1e6"])
    ps = hyd["psat"] * (1.0 + psis/hyd["RT_by_nu_by1e6"])
    pg = hyd["psat"] * (1.0 + psig/hyd["RT_by_nu_by1e6"])

    Rg    = (1.0/hyd["gg1"]) * hyd["RT_by_nu_by1e6"] * hyd["patm"] / hyd["psat"]
    Re    = (1.0/hyd["ge1"]) * hyd["RT_by_nu_by1e6"] * hyd["patm"] / hyd["psat"]
    Req1  = Re / (1.0 + Re/(hyd["Rsg"] + Rg))
    Rox   = f_Rox(psix)
    Rm    = (Rox - hyd["Rxm"]) / (1.0 + (Rox - hyd["Rxm"])/(hyd["Rms"] + Req1))
    gmH2O = (1.0/Rm) * hyd["RT_by_nu_by1e6"] * hyd["patm"] / hyd["psat"]

    DeltaPig_MPa = (hyd["Kin0"]+hyd["Clin0"]-Kin-Clin+hyd["otherIons"]) * hyd["R"]*hyd["T"] / 1e6
    Pg = f_P_from_psi(psig, hyd["epsilong"], hyd["pig0"], hyd["Pg0"], DeltaPig_MPa)
    Ps = f_P_from_psi(psis, hyd["epsilons"], hyd["pis0"], hyd["Ps0"], 0.0)
    gs = max(hyd["Xi"] * (Pg - hyd["m"]*Ps), 0.0)

    psii = f_SSC_apoplast_potential(psim, psig, psis, gs, gmH2O, pa, hyd, hyd["ge1"], hyd["gg1"])
    p_i  = hyd["psat"] * (1.0 + psii/hyd["RT_by_nu_by1e6"])

    Em  = gmH2O*      (pm - p_i) / hyd["patm"]
    Es1 = hyd["ge1"] *(ps - p_i) / hyd["patm"]
    Eg1 = hyd["gg1"] *(pg - p_i) / hyd["patm"]
    Es2 = hyd["ge2"] *(ps - pa)  / hyd["patm"]
    Eg2 = hyd["gg2"] *(pg - pa)  / hyd["patm"]

    Cm_c = hyd["Vm0"]/(hyd["epsilonm"]+hyd["pim0"])
    Cs_c = hyd["Vs0"]/(hyd["epsilons"]+hyd["pis0"])
    Cg_c = hyd["Vg0"]/(hyd["epsilong"]+hyd["pig0"])

    dpsimdt = (1/Cm_c)*(psix/hyd["Rxm"]+psim*(-1/hyd["Rxm"]-1/hyd["Rms"])+psis/hyd["Rms"]-Em)
    dpsisdt = (1/Cs_c)*(psim/hyd["Rms"]+psis*(-1/hyd["Rms"]-1/hyd["Rsg"])+psig/hyd["Rsg"]-Es1-Es2)
    dpsigdt = (1/Cg_c)*(psis/hyd["Rsg"]+psig*(-1/hyd["Rsg"])-Eg1-Eg2)

    # =========================================================================
    # MODULE 2: ABA BIOSYNTHESIS & CATABOLISM
    # =========================================================================
    DeltaPg_val = Pg - hyd["Pg0"]
    Vstress_val = f_Vstress(DeltaPg_val, hyd)

    dABAdt = (hyd["thetaABA"]
              * (Vstress_val + hyd["Vplus"]*ABA**hyd["n"]/(hyd["K1"]**hyd["n"]+ABA**hyd["n"]))
              - hyd["kcat"]*A8H*ABA/hyd["K2"])

    dA8Hdt = (hyd["thetaA8H"]*hyd["Vminus"]*ABA/(hyd["K3"]+ABA)
              - hyd["gammac"]*A8H)

    # =========================================================================
    # MODULE 3: ABA SIGNALING CASCADE
    # =========================================================================
    K1_nd = hyd["Km1"]/hyd["OST1T"]; K2_nd = hyd["Km2"]/hyd["OST1T"]
    K3_nd = hyd["Km3"]/hyd["SLAC1T"]; K4_nd = hyd["Km4"]/hyd["SLAC1T"]
    V1 = hyd["k1"]*hyd["MAP3KT"]; V2 = hyd["k2"]*hyd["PP2CT"]
    V3 = hyd["k3"]*hyd["OST1T"];  V4 = hyd["k4"]*hyd["PP2CT"]
    epsilon1 = hyd["MAP3KT"]/hyd["OST1T"]
    hx = {**hyd, "K1_nd":K1_nd, "K2_nd":K2_nd, "K3_nd":K3_nd, "K4_nd":K4_nd, "epsilon1":epsilon1}

    ABA_PYR = f_solve_ABA_PYR_quadratic(ABA, hyd)
    V2p = V2/(1.0+ABA_PYR/hyd["KI"]); V4p = V4/(1.0+ABA_PYR/hyd["KI"])
    K2prime = K2_nd*(1.0+So/K4_nd); K4prime = K4_nd*(1.0+Oo/K2_nd)

    O = f_OST1_roots_from_quadratic_non_dimensional(Oo, So, hx)
    dOodt = (V2p/hyd["OST1T"])*((V1/V2p)*O/(K1_nd+O) - Oo/(K2prime+Oo))

    S = f_SLAC1_conc_non_dimensional(Oo, So, hx)
    dSodt = (V4p/hyd["SLAC1T"])*((V3/V4p)*Oo*S/K3_nd - So/(K4prime+So))

    # =========================================================================
    # MODULE 4: GUARD CELL ELECTROPHYSIOLOGY
    # =========================================================================
    Clout = f_mass_balance(Clin, hyd["Clin0"], hyd["Clout0"], hyd)
    Kout  = f_mass_balance(Kin,  hyd["Kin0"],  hyd["Kout0"],  hyd)

    IGHK_Cl   = f_GHK_current_flux(hyd["PCl"], hyd["zCl"], Vmemb, Clin, Clout, hyd)
    ICl_SLAC1 = (hyd["SLAC1T"]/1e3)*So*IGHK_Cl

    p_KAT1  = f_p_open_KAT1_channel(Vmemb, hyd["V1by2_KAT1"], hyd["delta_KAT1"], hyd)
    IGHK_K  = f_GHK_current_flux(hyd["PK"], hyd["zK"], Vmemb, Kin, Kout, hyd)
    IK_KAT1 = hyd["NKAT1"]*p_KAT1*IGHK_K

    V12G    = f_V1by2_GORK(Kout, hyd)
    p_GORK  = f_p_open_GORK_channel(Vmemb, V12G, hyd["delta_GORK"], hyd)
    IK_GORK = hyd["NGORK"]*p_GORK*IGHK_K

    p_pump  = f_p_on_Hpump(ABA, hyd)
    IH_pump = f_HPump_current(p_pump, hyd["Hin0"], hyd["Hout0"], Vmemb, hyd)
    ICl_sym = f_2HClSymport_current(1.0, hyd["Hin0"], hyd["Hout0"], Clin, Clout, Vmemb, hyd)

    dVmembdt = -(1/hyd["Cm"])*(IH_pump+ICl_SLAC1+IK_KAT1+IK_GORK+ICl_sym)
    dClindt  = -(ICl_SLAC1-ICl_sym)*hyd["zCl"]*hyd["area"]/(hyd["F"]*hyd["vol_in"])
    dKindt   = -(IK_GORK+IK_KAT1)*hyd["zK"]*hyd["area"]/(hyd["F"]*hyd["vol_in"])

    return [dpsimdt, dpsisdt, dpsigdt, dABAdt, dA8Hdt,
            dOodt, dSodt, dVmembdt, dClindt, dKindt]


def evaluate_output_osc(t, y, hyd):
    """Post-process ODE solution into physically meaningful quantities.
    
    Returns dict with time, ABA, gs, RH, turgor pressures, and VPD.
    Time axis (t_plt) is centered on the first VPD step.
    """
    psig = y[:,2]; psis = y[:,1]; Clin = y[:,8]; Kin = y[:,9]
    ABA = y[:,3]; A8H = y[:,4]; Oo = y[:,5]; So = y[:,6]; Vmemb = y[:,7]
    t_plt = t - hyd["tstep1"]

    DeltaPig_MPa = (hyd["Kin0"]+hyd["Clin0"]-Kin-Clin+hyd["otherIons"]) * hyd["R"]*hyd["T"] / 1e6
    Pg = f_P_from_psi(psig, hyd["epsilong"], hyd["pig0"], hyd["Pg0"], DeltaPig_MPa)
    Ps = f_P_from_psi(psis, hyd["epsilons"], hyd["pis0"], hyd["Ps0"], 0.0)
    gs = np.maximum(hyd["Xi"]*(Pg - hyd["m"]*Ps), 0.0)
    RH  = f_RH_3step(t, hyd).astype(float)
    VPD = f_VPD_3step(t, hyd).astype(float)

    return dict(t=t, t_plt=t_plt, ABA=ABA, gs=gs, RH=RH, Pg=Pg, Ps=Ps, VPD=VPD)


def run_fullModel_osc(hyd):
    """Integrate the full 10-variable ODE for oscillation scenario.
    
    Uses BDF solver. Output sampled every 30 seconds.
    """
    tspan = np.arange(hyd["tstart"], hyd["tend"]+30, 30.0)
    y0 = [0.0, 0.0, 0.0, 1e-3, 1e-3, 0.0, 0.0, -180e-3, hyd["Clin0"], hyd["Kin0"]]
    sol = solve_ivp(
        lambda t,y: fullModel_DeltaPg_ode_osc(t, y, hyd),
        [hyd["tstart"], hyd["tend"]],
        y0, method="BDF", t_eval=tspan, rtol=1e-6, atol=1e-9)
    return evaluate_output_osc(sol.t, sol.y.T, hyd)


## 5. Simulation Engine

In [5]:
def run_figure3(hyd_overrides=None):
    hyd = read_parameters_oscillations()
    if hyd_overrides:
        # Extract direct ABA rates (bypass phi-derived recomputation)
        direct_rates = {}
        for k in ("Vstress", "Vplus", "Vminus"):
            if k in hyd_overrides:
                direct_rates[k] = hyd_overrides[k]

        # Extract combined K+=K- override
        K_pm_val = hyd_overrides.get("K_pm", None)

        for k, v in hyd_overrides.items():
            if k not in ("Vstress", "Vplus", "Vminus", "K_pm"):
                hyd[k] = v

        # Recompute derived quantities
        hyd["K1"] = hyd["lambda1"] * hyd["K2"]
        hyd["K3"] = hyd["lambda3"] * hyd["K2"]

        # Apply combined K+=K-
        if K_pm_val is not None:
            hyd["K1"] = K_pm_val
            hyd["K3"] = K_pm_val

        hyd["pig0"] = hyd["Pg0"] + 0.1
        hyd["pis0"] = hyd["Ps0"] + 0.1
        hyd["pim0"] = hyd["Pm0"] + 0.1
        hyd["vol_in"] = hyd["Vg0"] * hyd["nu_w"] * 1e-3 / hyd["stomatal_density"]
        hyd["VPD1"] = hyd["psat"] * (1 - hyd["RH1"]/100)
        hyd["VPD2"] = hyd["psat"] * (1 - hyd["RH2"]/100)
        hyd["VPD3"] = hyd["psat"] * (1 - hyd["RH3"]/100)

        if direct_rates:
            hyd["Vstress"] = direct_rates.get("Vstress", hyd["Vstress"])
            hyd["Vplus"]   = direct_rates.get("Vplus",   hyd["Vplus"])
            hyd["Vminus"]  = direct_rates.get("Vminus",  hyd["Vminus"])
        else:
            # NOTE: Figure 3 Vminus formula differs!
            hyd["Vminus"] = (hyd["K2"] * hyd["gammac"]**2) / (hyd["kcat"] * hyd["phi3"])
            hyd["Vplus"]  = hyd["phi2"] * hyd["kcat"] * hyd["Vminus"] / hyd["gammac"]
            hyd["Vstress"]= hyd["phi1"] * hyd["kcat"] * hyd["Vminus"] / hyd["gammac"]

    return run_fullModel_osc(hyd)

print('Figure 3 simulation engine ready.')

Figure 3 simulation engine ready.


## 6. Baseline Run (Default Parameters)

In [6]:
print("Running baseline oscillation (this takes ~30-60 seconds)...")
baseline = run_figure3()
print("Baseline complete.")

Running baseline oscillation (this takes ~30-60 seconds)...
Baseline complete.


## Figure 3: Oscillation Dynamics

Publication-quality plots showing the model's oscillatory behavior under
3-step RH protocol (80% → 55% → 30%) with oscillation-specific parameters.

In [ ]:
# ── Figure 3b: RH, ABA, gs vs time (oscillation dynamics) ────────────────────
xlim_L_osc, xlim_R_osc = -50, 200

fig_oscillations, axes = plt.subplots(3, 1, figsize=(12, 12))
t_min = baseline["t_plt"] / 60

# Panel 1: RH
ax = axes[0]
ax.plot(t_min, baseline['RH'], '-k', linewidth=2.5)
ax.set_xlim(xlim_L_osc, xlim_R_osc); ax.set_ylim(0, 100)
ax.set_ylabel('RH (%)', fontsize=16)
ax.tick_params(axis='both', which='major', labelsize=13)
ax.minorticks_on(); ax.grid(True, which='both', alpha=0.3)
ax.set_title('Figure 3: Oscillation Dynamics (WT)', fontsize=16, fontweight='bold')

# Panel 2: ABA
ax = axes[1]
ax.plot(t_min, baseline['ABA'], '-k', linewidth=2.5)
ax.set_xlim(xlim_L_osc, xlim_R_osc); ax.set_ylim(0, 3000)
ax.set_ylabel(r'[ABA] (nM)', fontsize=16)
ax.tick_params(axis='both', which='major', labelsize=13)
ax.minorticks_on(); ax.grid(True, which='both', alpha=0.3)

# Panel 3: gs
ax = axes[2]
ax.plot(t_min, baseline['gs'], '-k', linewidth=2.5)
ax.set_xlim(xlim_L_osc, xlim_R_osc); ax.set_ylim(0, 200)
ax.set_xlabel('Time elapsed (min)', fontsize=16)
ax.set_ylabel(r'$g_s$ (mmol m$^{-2}$ s$^{-1}$)', fontsize=16)
ax.tick_params(axis='both', which='major', labelsize=13)
ax.minorticks_on(); ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Dashboard Infrastructure

In [ ]:
from bokeh.plotting import figure
from bokeh.layouts import column as bokeh_column
from bokeh.models import HoverTool
import panel as pn
pn.extension("bokeh")

def make_figure3_plot(result, title_suffix):
    t_min = result["t_plt"] / 60
    t_base = baseline["t_plt"] / 60

    p1 = figure(title=f'RH — {title_suffix}', x_axis_label='Time (min)', y_axis_label='RH (%)',
                width=900, height=200, x_range=(-50, 200), y_range=(0, 100))
    p1.line(list(t_min), list(result["RH"]), line_width=2.5, color='black')

    p2 = figure(title=f'[ABA] — {title_suffix}', x_axis_label='Time (min)', y_axis_label='[ABA] (nM)',
                width=900, height=250, x_range=(-50, 200))
    p2.line(list(t_base), list(baseline["ABA"]), line_width=1.5, color='gray', alpha=0.5, legend_label='Baseline')
    p2.line(list(t_min), list(result["ABA"]), line_width=2.5, color='red', legend_label='Modified')
    p2.legend.location = 'top_right'

    p3 = figure(title=f'gs — {title_suffix}', x_axis_label='Time (min)', y_axis_label='gs (mmol/m²/s)',
                width=900, height=250, x_range=(-50, 200), y_range=(0, 200))
    p3.line(list(t_base), list(baseline["gs"]), line_width=1.5, color='gray', alpha=0.5, legend_label='Baseline')
    p3.line(list(t_min), list(result["gs"]), line_width=2.5, color='blue', legend_label='Modified')
    p3.legend.location = 'top_right'

    return bokeh_column(p1, p2, p3)

def make_dashboard(title, description, slider_specs, dashboard_name):
    """Create a parameter-exploration dashboard.
    
    slider_specs: list of tuples. Each tuple is either:
        (key, label, lo, hi, default, step)              — no unit conversion
        (key, label, lo, hi, default, step, scale_factor) — slider values are DIVIDED
            by scale_factor before passing to the ODE.
    """
    sliders = {}
    slider_widgets = []
    scale_factors = {}
    for spec in slider_specs:
        if len(spec) == 7:
            key, label, lo, hi, default, step_sz, sf = spec
            scale_factors[key] = sf
        else:
            key, label, lo, hi, default, step_sz = spec
            scale_factors[key] = 1.0
        s = pn.widgets.FloatSlider(name=label, start=lo, end=hi, value=default, step=step_sz, width=400)
        sliders[key] = s
        slider_widgets.append(s)

    status = pn.widgets.StaticText(value='Ready. Adjust sliders then click Compute.')
    compute_btn = pn.widgets.Button(name='Compute', button_type='success', width=200)
    reset_btn = pn.widgets.Button(name='Reset Defaults', button_type='warning', width=200)

    plot_pane = pn.pane.Bokeh(make_figure3_plot(baseline, f'{dashboard_name} (baseline)'))

    def on_compute(event):
        status.value = 'Computing... (this may take 30-60 seconds)'
        overrides = {}
        for k, s in sliders.items():
            overrides[k] = s.value / scale_factors[k]
        try:
            result = run_figure3(overrides)
            plot_pane.object = make_figure3_plot(result, dashboard_name)
            status.value = 'Done!'
        except Exception as e:
            status.value = f'Error: {e}'

    def on_reset(event):
        for spec in slider_specs:
            key, default = spec[0], spec[4]
            sliders[key].value = default
        status.value = 'Sliders reset to defaults.'

    compute_btn.on_click(on_compute)
    reset_btn.on_click(on_reset)

    n = len(slider_widgets)
    mid = (n + 1) // 2
    col1 = pn.Column(*slider_widgets[:mid])
    col2 = pn.Column(*slider_widgets[mid:])
    slider_row = pn.Row(col1, col2)

    header = pn.pane.Markdown(f'### {title}\n{description}')
    buttons = pn.Row(compute_btn, reset_btn)

    return pn.Column(header, slider_row, buttons, status, plot_pane)

print('Dashboard infrastructure ready.')

---
## Dashboard 1: Hydraulics

These parameters control **water transport** from xylem through mesophyll to guard cells, and the force balance that converts turgor to stomatal opening.

In [9]:
dashboard_hydraulics = make_dashboard(
    title='Hydraulics Parameters',
    description='Adjust tissue resistances and stomatal conductance equation coefficients.',
    slider_specs=[
        # (key,    label,                      min,   max,   default, step)
        ('Rms',   'R_ms (MPa/mmol/m²/s)',         0.5,  20.0,   5.0,    0.5),
        ('Rsg',   'R_sg (MPa/mmol/m²/s)',         1.0,  30.0,  10.0,    0.5),
        ('gg1',   'g_g¹ (mmol/m²/s)',     0.5,  10.0,   2.9,    0.1),
        ('gg2',   'g_g² (mmol/m²/s)',            0.2,   3.0,   1.0,    0.1),
        ('ge1',   'g_e¹ (mmol/m²/s)',     0.5,  10.0,   2.9,    0.1),
        ('ge2',   'g_e² (mmol/m²/s)',            1.0,  30.0,  10.0,    0.5),
    ],
    dashboard_name='Hydraulics'
)
dashboard_hydraulics


Column
    [0] Markdown(str)
    [1] Row
        [0] Column
            [0] FloatSlider(end=20.0, name='R_ms (MPa/mmol/m²/s)', start=0.5, step=0.5, value=5.0, width=400)
            [1] FloatSlider(end=30.0, name='R_sg (MPa/mmol/m²/s)', start=1.0, step=0.5, value=10.0, width=400)
            [2] FloatSlider(end=10.0, name='g_g¹ (mmol/m²/s)', start=0.5, value=2.9, width=400)
        [1] Column
            [0] FloatSlider(end=3.0, name='g_g² (mmol/m²/s)', start=0.2, value=1.0, width=400)
            [1] FloatSlider(end=10.0, name='g_e¹ (mmol/m²/s)', start=0.5, value=2.9, width=400)
            [2] FloatSlider(end=30.0, name='g_e² (mmol/m²/s)', start=1.0, step=0.5, value=10.0, width=400)
    [2] Row
        [0] Button(button_type='success', name='Compute', width=200)
        [1] Button(button_type='warning', name='Reset Defaults', width=200)
    [3] StaticText(value='Ready. Adjust s...)
    [4] Bokeh(Column)

---
## Dashboard 2: Stress Sensing (Vstress → ΔPg Curve)

These parameters shape the **stress-responsive ABA synthesis pathway**, modulating how guard cell turgor loss triggers ABA production.

In [10]:
dashboard_vstress = make_dashboard(
    "Vstress \u2192 \u0394Pg Curve",
    "Stress-sensing sigmoid shape parameters.",
    slider_specs=[
        ('vstress_exponent', 'Hill exponent n',     1.0,  6.0,  3.25,  0.25),
        ('vstress_khalf',    'K_half (MPa\u207f)',   0.2,  5.0,  1.4,   0.1),
    ],
    dashboard_name="Vstress Curve"
)
dashboard_vstress


Column
    [0] Markdown(str)
    [1] Row
        [0] Column
            [0] FloatSlider(end=6.0, name='Hill exponent n', start=1.0, step=0.25, value=3.25, width=400)
        [1] Column
            [0] FloatSlider(end=5.0, name='K_half (MPaⁿ)', start=0.2, value=1.4, width=400)
    [2] Row
        [0] Button(button_type='success', name='Compute', width=200)
        [1] Button(button_type='warning', name='Reset Defaults', width=200)
    [3] StaticText(value='Ready. Adjust s...)
    [4] Bokeh(Column)

---
## Dashboard 3: ABA Biosynthesis & Catabolism

These parameters control the **production and breakdown of ABA** via the autoregulation motif (SI Section S3.1, Eqs. S5, S8): stress-induced synthesis ($V_{stress}$), positive feedback ($V_+$, $K_+$), enzymatic degradation by A8H ($V_-$, $K_d$, $\gamma_A$), and A8H turnover ($K_-$, $\gamma_H$).

In [11]:
dashboard_aba_bio = make_dashboard(
    title='ABA Biosynthesis & Catabolism',
    description='Synthesis rates, feedback gains, and enzyme kinetics of the ABA autoregulatory motif.',
    slider_specs=[
        # key        label                              min     max     default   step   [scale]
        ('Vstress', 'V_stress (nM/s)',                   1.0,   30.0,    8.85,   0.1),
        ('Vplus',   'V₊ (nM/s)',                        10.0,  200.0,   67.58,   1.0),
        ('Vminus',  'V₋ (nM/s)',                         0.1,   10.0,    1.51,   0.05),
        ('K_pm',    'K₊ = K₋ (nM)',                    200.0, 5000.0, 1560.0,  20.0),
        ('K2',      'K_d (nM)',                        500.0, 5000.0, 2000.0,  50.0),
        ('kcat',    'γ_A (1/s)',                         0.05,    1.0,   0.25,   0.01),
        ('gammac',  'γ_H (×10⁻³ 1/s)',                  0.5,   10.0,   3.072,   0.1,  1e3),
        ('n',       'n (Hill coefficient)',               1.0,    5.0,    3.0,   1.0),
    ],
    dashboard_name='ABA Biosynthesis'
)
dashboard_aba_bio


Column
    [0] Markdown(str)
    [1] Row
        [0] Column
            [0] FloatSlider(end=30.0, name='V_stress (nM/s)', start=1.0, value=8.85, width=400)
            [1] FloatSlider(end=200.0, name='V₊ (nM/s)', start=10.0, step=1.0, value=67.58, width=400)
            [2] FloatSlider(end=10.0, name='V₋ (nM/s)', start=0.1, step=0.05, value=1.51, width=400)
            [3] FloatSlider(end=5000.0, name='K₊ = K₋ (nM)', start=200.0, step=20.0, value=1560.0, width=400)
        [1] Column
            [0] FloatSlider(end=5000.0, name='K_d (nM)', start=500.0, step=50.0, value=2000.0, width=400)
            [1] FloatSlider(name='γ_A (1/s)', start=0.05, step=0.01, value=0.25, width=400)
            [2] FloatSlider(end=10.0, name='γ_H (×10⁻³ 1/s)', start=0.5, value=3.072, width=400)
            [3] FloatSlider(end=5.0, name='n (Hill coefficient)', start=1.0, step=1.0, value=3.0, width=400)
    [2] Row
        [0] Button(button_type='success', name='Compute', width=200)
        [1] Button(button_type='warning', name='Reset Defaults', width=200)
    [3] StaticText(value='Ready. Adjust s...)
    [4] Bokeh(Column)

---
## Dashboard 4: ABA Signaling Cascade

These parameters govern the **receptor-kinase-phosphatase cascade** (SI Section S3.2, Eqs. S28, S33) that translates cytosolic ABA into SLAC1 ion channel activation via the OST1 bicyclic futile cycle with non-competitive PP2C inhibition.

In [12]:
dashboard_aba_sig = make_dashboard(
    title='ABA Signaling Cascade',
    description='Receptor binding, kinase/phosphatase rates, Michaelis constants, and protein levels.',
    slider_specs=[
        # Kinase / phosphatase rates
        # key    label                              min     max      default    step    [scale]
        ('k1',   'k₁ MAP3K→OST1 (1/s)',           0.005,   0.1,    0.03,     0.005),
        ('k2',   'k₂ PP2C→OST1 (1/s)',            0.005,   0.1,    0.033,    0.001),
        ('k3',   'k₃ OST1*→SLAC1 (1/s)',          0.5,    10.0,    3.75,     0.25),
        ('k4',   'k₄ PP2C→SLAC1 (1/s)',           0.02,    0.5,    0.139,    0.005),
        # Michaelis constants
        ('Km1',  'K_m1 OST1 phosph. (nM)',        500.0, 8000.0, 2500.0,   100.0),
        ('Km2',  'K_m2 OST1 dephosph. (nM)',       10.0,  500.0,   89.7,     5.0),
        ('Km3',  'K_m3 SLAC1 phosph. (µM)',         1.0,   50.0,   18.93,    0.5,  1e-3),
        ('Km4',  'K_m4 SLAC1 dephosph. (nM)',      50.0, 2000.0,  597.0,    10.0),
        # Receptor / inhibition
        ('KD',   'K_D ABA–PYR dissoc. (nM)',      200.0, 3000.0, 1000.0,    50.0),
        ('KI',   'K_I PP2C inhibition (nM)',         5.0,  100.0,   30.0,     1.0),
        # Protein totals
        ('OST1T',  '[OST1]_T (nM)',               200.0, 3000.0, 1000.0,   50.0),
        ('PP2CT',  '[PP2C]_T (nM)',               200.0, 3000.0, 1000.0,   50.0),
        ('SLAC1T', '[SLAC1]_T (nM)',              200.0, 3000.0, 1000.0,   50.0),
        ('MAP3KT',  '[MAP3K]_T (nM)',              200.0, 3000.0, 1000.0,   50.0),
    ],
    dashboard_name='ABA Signaling'
)
dashboard_aba_sig


Column
    [0] Markdown(str)
    [1] Row
        [0] Column
            [0] FloatSlider(end=0.1, name='k₁ MAP3K→OST1 (1/s)', start=0.005, step=0.005, value=0.03, width=400)
            [1] FloatSlider(end=0.1, name='k₂ PP2C→OST1 (1/s)', start=0.005, step=0.001, value=0.033, width=400)
            [2] FloatSlider(end=10.0, name='k₃ OST1*→SLAC1 (1/s)', start=0.5, step=0.25, value=3.75, width=400)
            [3] FloatSlider(end=0.5, name='k₄ PP2C→SLAC1 (1/s)', start=0.02, step=0.005, value=0.139, width=400)
            [4] FloatSlider(end=8000.0, name='K_m1 OST1 phosph. (nM)', start=500.0, step=100.0, value=2500.0, width=400)
            [5] FloatSlider(end=500.0, name='K_m2 OST1 dephosph. (..., start=10.0, step=5.0, value=89.7, width=400)
            [6] FloatSlider(end=50.0, name='K_m3 SLAC1 phosph. (µM)', start=1.0, step=0.5, value=18.93, width=400)
        [1] Column
            [0] FloatSlider(end=2000.0, name='K_m4 SLAC1 dephosph. (..., start=50.0, step=10.0, value=597.0, width=400)
            [1] FloatSlider(end=3000.0, name='K_D ABA–PYR d..., start=200.0, step=50.0, value=1000.0, width=400)
            [2] FloatSlider(end=100.0, name='K_I PP2C inhibition (..., start=5.0, step=1.0, value=30.0, width=400)
            [3] FloatSlider(end=3000.0, name='[OST1]_T (nM)', start=200.0, step=50.0, value=1000.0, width=400)
            [4] FloatSlider(end=3000.0, name='[PP2C]_T (nM)', start=200.0, step=50.0, value=1000.0, width=400)
            [5] FloatSlider(end=3000.0, name='[SLAC1]_T (nM)', start=200.0, step=50.0, value=1000.0, width=400)
            [6] FloatSlider(end=3000.0, name='[MAP3K]_T (nM)', start=200.0, step=50.0, value=1000.0, width=400)
    [2] Row
        [0] Button(button_type='success', name='Compute', width=200)
        [1] Button(button_type='warning', name='Reset Defaults', width=200)
    [3] StaticText(value='Ready. Adjust s...)
    [4] Bokeh(Column)

---
## Dashboard 5: Electrophysiology

These parameters control **ion channels and pumps**, determining the electrical properties and ion fluxes in the guard cell.

In [13]:
dashboard_electro = make_dashboard(
    "Electrophysiology",
    "Ion concentrations and ABA-pump sensitivity governing guard cell membrane potential.",
    slider_specs=[
        ('Clin0',           'Cl\u207b_in (mM)',    50.0, 500.0,  260.0,  10.0),
        ('Clout0',          'Cl\u207b_out (mM)',     5.0, 100.0,   22.0,   1.0),
        ('Kin0',            'K\u207a_in (mM)',     50.0, 500.0,  210.0,  10.0),
        ('Kout0',           'K\u207a_out (mM)',      5.0, 100.0,   20.0,   1.0),
        ('ABA_Hpump_Khalf', 'K_ABA (nM)',           10.0, 200.0,   50.0,   5.0),
        # --- GORK channel ---
        ('delta_GORK',       'δ_GORK',               0.5,   5.0,    2.0,   0.1),
        ('V1by2_Kconc_GORK', 'V½_GORK K⁺ (mM)',     10.0, 500.0,  150.0,  10.0),
        # --- KAT1 channel ---
        ('delta_KAT1',       'δ_KAT1',               0.5,   5.0,    1.6,   0.1),
        ('V1by2_KAT1',       'V½_KAT1 (mV)',       -300.0, -50.0, -170.0,   5.0, 1000.0),
        # --- H⁺-ATPase pump ---
        ('DeltaGATP',        'ΔG_ATP (kJ/mol)',     -40.0, -15.0,  -26.0,   0.5, 0.001),
        ('E0pump',           'E₀_pump (×10⁶)',        0.1,   5.0,  1.1033,  0.0001, 1e-6),
        ('k1pump',           'k₁_pump',              0.01,   2.0,   0.55,  0.01),
        ('k2pump',           'k₂_pump (×10⁻⁴)',      0.1,  10.0,   2.58,   0.01, 1e4),
        # --- Symporter / SLAC1 scaling ---
        ('V0Cl',             'V₀_Cl (×10⁴)',         1.0,  30.0,   9.25,  0.25, 1e-4),
        # --- ABA pump inhibition ---
        ('ABA_Hpump_n',      'n_ABA pump',            0.5,   4.0,    1.0,   0.5),
    ],
    dashboard_name="Electrophysiology"
)
dashboard_electro


Column
    [0] Markdown(str)
    [1] Row
        [0] Column
            [0] FloatSlider(end=500.0, name='Cl⁻_in (mM)', start=50.0, step=10.0, value=260.0, width=400)
            [1] FloatSlider(end=100.0, name='Cl⁻_out (mM)', start=5.0, step=1.0, value=22.0, width=400)
            [2] FloatSlider(end=500.0, name='K⁺_in (mM)', start=50.0, step=10.0, value=210.0, width=400)
            [3] FloatSlider(end=100.0, name='K⁺_out (mM)', start=5.0, step=1.0, value=20.0, width=400)
            [4] FloatSlider(end=200.0, name='K_ABA (nM)', start=10.0, step=5.0, value=50.0, width=400)
            [5] FloatSlider(end=5.0, name='δ_GORK', start=0.5, value=2.0, width=400)
            [6] FloatSlider(end=500.0, name='V½_GORK K⁺ (mM)', start=10.0, step=10.0, value=150.0, width=400)
            [7] FloatSlider(end=5.0, name='δ_KAT1', start=0.5, value=1.6, width=400)
        [1] Column
            [0] FloatSlider(end=-50.0, name='V½_KAT1 (mV)', start=-300.0, step=5.0, value=-170.0, width=400)
            [1] FloatSlider(end=-15.0, name='ΔG_ATP (kJ/mol)', start=-40.0, step=0.5, value=-26.0, width=400)
            [2] FloatSlider(end=5.0, name='E₀_pump (×10⁶)', start=0.1, step=0.0001, value=1.1033, width=400)
            [3] FloatSlider(end=2.0, name='k₁_pump', start=0.01, step=0.01, value=0.55, width=400)
            [4] FloatSlider(end=10.0, name='k₂_pump (×10⁻⁴)', start=0.1, step=0.01, value=2.58, width=400)
            [5] FloatSlider(end=30.0, name='V₀_Cl (×10⁴)', start=1.0, step=0.25, value=9.25, width=400)
            [6] FloatSlider(end=4.0, name='n_ABA pump', start=0.5, step=0.5, value=1.0, width=400)
    [2] Row
        [0] Button(button_type='success', name='Compute', width=200)
        [1] Button(button_type='warning', name='Reset Defaults', width=200)
    [3] StaticText(value='Ready. Adjust s...)
    [4] Bokeh(Column)

In [ ]:
# Build the page that will be served by panel convert.
pn.Column(
    "# Stomatal oscillations dashboard",
    "Three-step humidity oscillation protocol (80% → 55% → 30% RH).",
    pn.Tabs(
        ("Hydraulics",        dashboard_hydraulics),
        ("V_stress",          dashboard_vstress),
        ("ABA biosynthesis",  dashboard_aba_bio),
        ("ABA signaling",     dashboard_aba_sig),
        ("Electrophysiology", dashboard_electro),
    ),
).servable()
